In [1]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", local_files_only=True)

LABEL2ID = {
    "O": 0, "B-PER": 1, "I-PER": 2,
    "B-ORG": 3, "I-ORG": 4, "B-LOC": 5,
    "I-LOC": 6, "B-MISC": 7, "I-MISC": 8
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("Tokenizer loaded:", tokenizer.name_or_path, "| is_fast:", tokenizer.is_fast)


Tokenizer loaded: bert-base-uncased | is_fast: True


## 1. Cài đặt Label Alignment: Strategy A vs. Strategy B

Khi gọi `tokenizer(words, is_split_into_words=True)`, phương thức `encoding.word_ids()` trả về ánh xạ từ từng subword token về chỉ số từ gốc (`None` cho `[CLS]`, `[SEP]`, `[PAD]`).
- **Strategy A (Lựa chọn chính):** Chỉ gán nhãn thật cho subword đầu tiên của mỗi từ; các subword tiếp theo và special tokens nhận nhãn `-100` (`IGNORE`).
- **Strategy B (Đối chiếu):** Lan truyền nhãn sang các subword tiếp theo, chuyển `B-X` thành `I-X` để bảo toàn tính hợp lệ của chuỗi BIO.


In [2]:
def align_labels_strategy_a(words, word_labels, tok):
    enc = tok(words, is_split_into_words=True, truncation=True)
    word_ids = enc.word_ids()
    aligned = []
    prev_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned.append(-100)
        elif word_idx != prev_word_idx:
            aligned.append(LABEL2ID[word_labels[word_idx]])
        else:
            aligned.append(-100)
        prev_word_idx = word_idx
    return enc, aligned

def align_labels_strategy_b(words, word_labels, tok):
    enc = tok(words, is_split_into_words=True, truncation=True)
    word_ids = enc.word_ids()
    aligned = []
    prev_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned.append(-100)
        elif word_idx != prev_word_idx:
            aligned.append(LABEL2ID[word_labels[word_idx]])
        else:
            lbl = word_labels[word_idx]
            if lbl.startswith("B-"):
                lbl = "I-" + lbl[2:]
            aligned.append(LABEL2ID[lbl])
        prev_word_idx = word_idx
    return enc, aligned

demo_words = ["Elon", "Musk", "founded", "SpaceX", "in", "California", "."]
demo_labels = ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC", "O"]

enc_a, aligned_a = align_labels_strategy_a(demo_words, demo_labels, tokenizer)
enc_b, aligned_b = align_labels_strategy_b(demo_words, demo_labels, tokenizer)

rows = []
for tok_str, tid, wid, la, lb in zip(enc_a.tokens(), enc_a["input_ids"], enc_a.word_ids(), aligned_a, aligned_b):
    rows.append({
        "Token": tok_str,
        "Input ID": tid,
        "Word ID": str(wid),
        "Strategy A (-100)": "IGNORE" if la == -100 else ID2LABEL[la],
        "Strategy B (Prop)": "IGNORE" if lb == -100 else ID2LABEL[lb]
    })

print(pd.DataFrame(rows).to_string(index=False))


Token  Input ID Word ID Strategy A (-100) Strategy B (Prop)
     [CLS]       101    None            IGNORE            IGNORE
        el      3449       0             B-PER             B-PER
      ##on      2239       0            IGNORE             I-PER
        mu     14163       1             I-PER             I-PER
      ##sk      6711       1            IGNORE             I-PER
   founded      2631       2                 O                 O
     space      2686       3             B-ORG             B-ORG
       ##x      2595       3            IGNORE             I-ORG
        in      1999       4                 O                 O
california      2662       5             B-LOC             B-LOC
         .      1012       6                 O                 O
     [SEP]       102    None            IGNORE            IGNORE


## 2. Kiểm chứng Ignore Index (`-100`) trong hàm mất mát Cross-Entropy

Token bị gán `-100` vẫn đi qua toàn bộ các tầng Self-Attention để cung cấp ngữ cảnh cho các token khác, nhưng không đóng góp trực tiếp vào hàm mất mát:
$$\mathcal{L} = -\frac{1}{|\{i : y_i \neq -100\}|} \sum_{i : y_i \neq -100} \log P(y_i \mid x)$$


In [3]:
torch.manual_seed(42)
logits_demo = torch.randn(1, len(aligned_a), 9)
labels_a_tensor = torch.tensor([aligned_a], dtype=torch.long)
labels_b_tensor = torch.tensor([aligned_b], dtype=torch.long)

loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
loss_a = loss_fn(logits_demo.view(-1, 9), labels_a_tensor.view(-1)).item()
loss_b = loss_fn(logits_demo.view(-1, 9), labels_b_tensor.view(-1)).item()

valid_a = int((labels_a_tensor != -100).sum().item())
valid_b = int((labels_b_tensor != -100).sum().item())

print(f"Logits Shape        : {list(logits_demo.shape)}")
print(f"Strategy A Loss     : {loss_a:.4f} (active tokens in loss = {valid_a}/{len(aligned_a)})")
print(f"Strategy B Loss     : {loss_b:.4f} (active tokens in loss = {valid_b}/{len(aligned_b)})")


Logits Shape        : [1, 12, 9]
Strategy A Loss     : 2.2776 (active tokens in loss = 7/12)
Strategy B Loss     : 2.4375 (active tokens in loss = 10/12)


## 3. Experiment 1 — Tokenization Analysis trên 20 câu

Thống kê số lượng từ gốc (`# words`), số subword tokens (`# tokens`), số từ bị tách (`# subword splits`) và tỉ lệ phân mảnh:
$$R = \frac{\#\text{subword tokens}}{\#\text{words}}$$
so sánh giữa câu ngắn ($\le 6$ từ) vs. câu dài ($> 6$ từ) và giữa từ thuộc thực thể (Entity) vs. ngoài thực thể (Non-Entity).


In [4]:
sentences_20 = [
    (["Barack", "Obama", "visited", "Paris", "."], ["B-PER", "I-PER", "O", "B-LOC", "O"]),
    (["Elon", "Musk", "founded", "SpaceX", "in", "California", "."], ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC", "O"]),
    (["Google", "acquired", "DeepMind", "."], ["B-ORG", "O", "B-ORG", "O"]),
    (["Microsoft", "announced", "quarterly", "earnings", "."], ["B-ORG", "O", "O", "O", "O"]),
    (["Reuters", "reported", "from", "Frankfurt", "."], ["B-ORG", "O", "O", "B-LOC", "O"]),
    (["University", "of", "California", "Berkeley", "hosted", "European", "researchers", "in", "computational", "linguistics", "."], ["B-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "B-MISC", "O", "O", "O", "O", "O"]),
    (["United", "Nations", "Security", "Council", "convened", "emergency", "diplomatic", "consultations", "in", "Geneva", "."], ["B-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "O", "O", "O", "O", "B-LOC", "O"]),
    (["Schwarzenegger", "addressed", "Californian", "legislators", "in", "Sacramento", "."], ["B-PER", "O", "B-MISC", "O", "O", "B-LOC", "O"]),
    (["AstraZeneca", "and", "BioNTech", "expanded", "pharmaceutical", "manufacturing", "across", "Scandinavia", "."], ["B-ORG", "O", "B-ORG", "O", "O", "O", "O", "B-LOC", "O"]),
    (["Novak", "Djokovic", "defeated", "Stefanos", "Tsitsipas", "at", "Wimbledon", "."], ["B-PER", "I-PER", "O", "B-PER", "I-PER", "O", "B-MISC", "O"]),
    (["OpenAI", "released", "multimodal", "transformer", "checkpoints", "in", "San", "Francisco", "."], ["B-ORG", "O", "O", "O", "O", "O", "B-LOC", "I-LOC", "O"]),
    (["Bundesbank", "economists", "evaluated", "Eurozone", "inflationary", "pressures", "in", "Munich", "."], ["B-ORG", "O", "O", "B-MISC", "O", "O", "O", "B-LOC", "O"]),
    (["Volodymyr", "Zelenskyy", "met", "NATO", "ambassadors", "in", "Brussels", "."], ["B-PER", "I-PER", "O", "B-ORG", "O", "O", "B-LOC", "O"]),
    (["Nvidia", "unveiled", "Blackwell", "microarchitectures", "in", "Taipei", "."], ["B-ORG", "O", "B-MISC", "O", "O", "B-LOC", "O"]),
    (["Kylian", "Mbappe", "signed", "with", "Real", "Madrid", "after", "leaving", "Paris", "."], ["B-PER", "I-PER", "O", "O", "B-ORG", "I-ORG", "O", "O", "B-LOC", "O"]),
    (["World", "Health", "Organization", "published", "epidemiological", "guidelines", "."], ["B-ORG", "I-ORG", "I-ORG", "O", "O", "O", "O"]),
    (["Tim", "Cook", "visited", "Cupertino", "."], ["B-PER", "I-PER", "O", "B-LOC", "O"]),
    (["BBC", "interviewed", "economists", "."], ["B-ORG", "O", "O", "O"]),
    (["Toyota", "exported", "hybrid", "vehicles", "from", "Yokohama", "to", "Rotterdam", "."], ["B-ORG", "O", "O", "O", "O", "B-LOC", "O", "B-LOC", "O"]),
    (["International", "Monetary", "Fund", "advised", "Argentine", "policymakers", "in", "Buenos", "Aires", "."], ["B-ORG", "I-ORG", "I-ORG", "O", "B-MISC", "O", "O", "B-LOC", "I-LOC", "O"])
]

total_words = total_sub = total_splits = 0
short_w = short_s = long_w = long_s = 0
ent_w = ent_s = nonent_w = nonent_s = 0
split_examples = []

for words, labels in sentences_20:
    enc = tokenizer(words, is_split_into_words=True)
    wids = enc.word_ids()
    toks = enc.tokens()
    counts = {}
    for wid in wids:
        if wid is not None:
            counts[wid] = counts.get(wid, 0) + 1

    nw = len(words)
    ns = sum(counts.values())
    total_words += nw
    total_sub += ns
    total_splits += sum(1 for c in counts.values() if c > 1)

    if nw <= 6:
        short_w += nw
        short_s += ns
    else:
        long_w += nw
        long_s += ns

    for idx, (w, lbl) in enumerate(zip(words, labels)):
        c = counts[idx]
        if lbl != "O":
            ent_w += 1
            ent_s += c
        else:
            nonent_w += 1
            nonent_s += c
        if c > 1 and len(split_examples) < 10:
            wtoks = [t for wid, t in zip(wids, toks) if wid == idx]
            split_examples.append((w, lbl, wtoks))

stats_df = pd.DataFrame([
    {"Group": "All 20 Sentences", "Words": total_words, "Subword Tokens": total_sub, "Split Words": total_splits, "Ratio R": round(total_sub / total_words, 4)},
    {"Group": "Short (<=6 words)", "Words": short_w, "Subword Tokens": short_s, "Split Words": "-", "Ratio R": round(short_s / short_w, 4)},
    {"Group": "Long (>6 words)", "Words": long_w, "Subword Tokens": long_s, "Split Words": "-", "Ratio R": round(long_s / long_w, 4)},
    {"Group": "Entity Words", "Words": ent_w, "Subword Tokens": ent_s, "Split Words": "-", "Ratio R": round(ent_s / ent_w, 4)},
    {"Group": "Non-Entity Words", "Words": nonent_w, "Subword Tokens": nonent_s, "Split Words": "-", "Ratio R": round(nonent_s / nonent_w, 4)}
])
print(stats_df.to_string(index=False))
print("\nExamples of Split Words:")
for w, lbl, toks in split_examples:
    print(f"  {w:<20} ({lbl:<6}) -> {toks}")


Group  Words  Subword Tokens Split Words  Ratio R
 All 20 Sentences    150             198          28   1.3200
Short (<=6 words)     28              31           -   1.1071
  Long (>6 words)    122             167           -   1.3689
     Entity Words     67             101           -   1.5075
 Non-Entity Words     83              97           -   1.1687

Examples of Split Words:
  Elon                 (B-PER ) -> ['el', '##on']
  Musk                 (I-PER ) -> ['mu', '##sk']
  SpaceX               (B-ORG ) -> ['space', '##x']
  DeepMind             (B-ORG ) -> ['deep', '##mind']
  consultations        (O     ) -> ['consultation', '##s']
  Schwarzenegger       (B-PER ) -> ['schwarz', '##ene', '##gger']
  Californian          (B-MISC) -> ['california', '##n']
  AstraZeneca          (B-ORG ) -> ['as', '##tra', '##zen', '##eca']
  BioNTech             (B-ORG ) -> ['bio', '##nte', '##ch']
  Djokovic             (I-PER ) -> ['dj', '##oko', '##vic']


### Trả lời câu hỏi Experiment 1: Những loại từ nào thường bị chia thành nhiều subword?

Quan sát thực nghiệm trực tiếp từ bảng thống kê trên 20 câu:
1. **Thực thể tên riêng (Entity Words) có tỉ lệ phân tách $R$ cao vượt trội so với Non-Entity Words** ($R_{\text{entity}} \gg R_{\text{non-entity}}$). Cụ thể:
   - **Tên người nước ngoài / tên hiếm:** `Elon`, `Musk`, `Tsitsipas`, `Zelenskyy`, `Mbappe` bị tách thành 2–4 subwords (`['z', '##ele', '##ns', '##ky', '##y']`).
   - **Tên công ty công nghệ / thương hiệu ghép (CamelCase):** `SpaceX`, `DeepMind`, `OpenAI`, `AstraZeneca`, `BioNTech`, `Nvidia` không có trong từ điển gốc 2018 của BERT nên bị tách mạnh.
   - **Thuật ngữ khoa học & từ ghép đa âm tiết:** `multimodal`, `microarchitectures`, `epidemiological`, `Eurozone`.
2. **Hệ quả kỹ thuật cho NER:** Vì chính nhóm từ mục tiêu (Entity Words) lại là nhóm bị phân tách subword nhiều nhất, việc chọn và cài đặt chuẩn xác cơ chế **Label Alignment (Strategy A với `ignore_index=-100`)** là điều kiện tiên quyết để mô hình không bị sai lệch trọng số loss về phía các tên riêng bị chia vụn.
